In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# ==========================================
# 0. SETUP: GENERATE RAW GL DATA
# ==========================================
RAW_GL = """txn_id,date,gl_code,category,description,amount,rep_id
T001,2024-01-05,REV-01,Revenue,Product Sales - North,4200,R01
T002,2024-01-12,REV-02,revenue,Service Contract,6750,R02
T003,2024-01-20,COG-01,COGS,Inventory Purchase,-1800,R01
T004,2024-01-28,OPX-01,opex,Office Rent,-3500,R02
T005,2024-02-03,REV-01,Revenue,Product Sales - South,$5500,R03
T006,2024-02-10,REV-03,Revenue,Licence Renewal,2950,R04
T007,2024-02-14,COG-02,cogs,Direct Labour,-2100,R03
T008,2024-02-22,OPX-02,OPEX,Salaries,-1400,R04
T009,2024-03-01,REV-01,Revenue,Product Sales - East,9200,R01
T010,2024-03-09,REV-02,revenue,Consulting Fees,3800,R04
T011,2024-03-15,COG-01,COGS,Raw Materials,-2400,R02
T012,2024-03-22,OPX-01,OpEx,Marketing Spend,-5200,R03
T013,2024-04-03,REV-01,Revenue,Product Sales,7100,R02
T014,2024-04-10,REV-03,revenue,Licence Renewal,4300,R03
T015,2024-04-18,COG-02,COGS,Direct Labour,-1900,R01
T016,2024-04-25,OPX-02,opex,Salaries,-2800,R04
T017,2024-05-02,REV-01,Revenue,Product Sales,8500,R01
T018,2024-05-14,REV-02,revenue,Service Contract,5100,R02
T019,2024-05-20,COG-01,cogs,Raw Materials,-2600,R03
T020,2024-05-28,OPX-01,OPEX,Office Rent,-3100,R04
T021,2024-06-05,REV-01,Revenue,Product Sales,9800,R03
T022,2024-06-12,REV-02,revenue,Consulting Fees,6200,R04
T023,2024-06-20,COG-02,COGS,Direct Labour,-3100,R01
T024,2024-06-28,OPX-02,opex,Salaries,-4200,R02
T001,2024-01-05,REV-01,Revenue,Product Sales - North,4200,R01"""

df_raw_initial = pd.read_csv(io.StringIO(RAW_GL))
df_raw_initial.to_csv('gl_data.csv', index=False)


# ==========================================
# TASK 1: THE CLEANING PIPELINE
# ==========================================
def clean_transactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates(subset=['txn_id'], keep='first').copy()

    cat_mapping = {'revenue': 'Revenue', 'cogs': 'COGS', 'opex': 'OpEx'}
    df['category'] = df['category'].str.lower().map(cat_mapping)

    unmapped = df['category'].isnull().sum()
    if unmapped > 0:
        print(f"    [!] WARNING: {unmapped} rows have unmapped categories")

    df['amount'] = df['amount'].astype(str).str.replace('$', '', regex=False).astype(float)
    df['date'] = pd.to_datetime(df['date'])

    return df.reset_index(drop=True)


# ==========================================
# TASK 2: P&L SUMMARY
# ==========================================
def build_pl_summary(df: pd.DataFrame) -> pd.DataFrame:
    # Resample to monthly
    monthly = (df.set_index('date')
                 .groupby('category')
                 .resample('ME')['amount'].sum()
                 .unstack('category')
                 .fillna(0)
                 .reset_index())

    monthly.columns.name = None
    monthly['month'] = monthly['date'].dt.strftime('%Y-%m')
    monthly = monthly.rename(columns={'Revenue': 'revenue', 'COGS': 'cogs', 'OpEx': 'opex'})

    # Derivations
    monthly['gross_profit'] = monthly['revenue'] + monthly['cogs']
    monthly['gp_margin_pct'] = (monthly['gross_profit'] / monthly['revenue'] * 100).round(1)
    monthly['ebit'] = monthly['gross_profit'] + monthly['opex']
    monthly['ebit_margin_pct'] = (monthly['ebit'] / monthly['revenue'] * 100).round(1)

    # Order columns
    cols = ['month', 'revenue', 'cogs', 'opex', 'gross_profit', 'gp_margin_pct', 'ebit', 'ebit_margin_pct']
    monthly = monthly[cols]

    # Append H1 Total Row
    h1_row = pd.DataFrame([
        {
            'month': 'H1 Total',
            'revenue': monthly['revenue'].sum(),
            'cogs': monthly['cogs'].sum(),
            'opex': monthly['opex'].sum(),
            'gross_profit': monthly['gross_profit'].sum(),
            'ebit': monthly['ebit'].sum()
        }
    ])
    h1_row['gp_margin_pct'] = (h1_row['gross_profit'] / h1_row['revenue'] * 100).round(1)
    h1_row['ebit_margin_pct'] = (h1_row['ebit'] / h1_row['revenue'] * 100).round(1)

    return pd.concat([monthly, h1_row], ignore_index=True)


# ==========================================
# TASK 3: COMPUTE KPIs
# ==========================================
def compute_kpis(pl_df: pd.DataFrame) -> dict:
    monthly = pl_df[pl_df['month'] != 'H1 Total'].copy()
    h1_rev  = monthly['revenue'].sum()
    h1_cogs = monthly['cogs'].sum()
    h1_ebit = monthly['ebit'].sum()

    def gm_pct(r, c): return round((r + c) / r * 100, 1) if r else None
    def em_pct(r, e): return round(e / r * 100, 1) if r else None
    def rev_growth(curr, prior): return round((curr - prior) / prior * 100, 1) if prior else None

    return {
        'H1 Revenue':       round(h1_rev, 0),
        'H1 EBIT':          round(h1_ebit, 0),
        'Gross Margin %':   gm_pct(h1_rev, h1_cogs),
        'EBIT Margin %':    em_pct(h1_rev, h1_ebit),
        'Best Month':       monthly.loc[monthly['revenue'].idxmax(), 'month'],
        'Worst Month':      monthly.loc[monthly['revenue'].idxmin(), 'month'],
        'Revenue Growth %': rev_growth(monthly['revenue'].iloc[-1], monthly['revenue'].iloc[0])
    }


# ==========================================
# TASK 4: CHARTS
# ==========================================
def build_charts(pl_df: pd.DataFrame) -> dict:
    monthly = pl_df[pl_df['month'] != 'H1 Total'].copy()
    charts = {}

    # Chart 1: Revenue Trend
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=monthly, x='month', y='revenue', marker='o', color='#1D9E75', linewidth=2)
    plt.title('H1 Revenue Trend: Steady Growth Through Q2', fontweight='bold', pad=15)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('chart_revenue_trend.png', dpi=150)
    plt.close()
    charts['trend'] = 'chart_revenue_trend.png'

    # Chart 2: EBIT vs GP%
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax2 = ax1.twinx()
    sns.barplot(data=monthly, x='month', y='ebit', color='#2D2D2D', ax=ax1, alpha=0.8)
    sns.lineplot(data=monthly, x='month', y='gp_margin_pct', color='#1D9E75', marker='o', ax=ax2, linewidth=2.5)
    ax1.set_title('EBIT Growth Driven by Sustained Gross Margins', fontweight='bold', pad=15)
    ax1.set_ylabel('EBIT ($)')
    ax2.set_ylabel('Gross Margin (%)')
    plt.tight_layout()
    plt.savefig('chart_ebit_margin.png', dpi=150)
    plt.close()
    charts['dual_axis'] = 'chart_ebit_margin.png'

    # Chart 3: P&L Waterfall
    h1_totals = pl_df[pl_df['month'] == 'H1 Total'].iloc[0]
    metrics = ['Revenue', 'COGS', 'Gross Profit', 'OpEx', 'EBIT']
    values = [h1_totals['revenue'], h1_totals['cogs'], h1_totals['gross_profit'], h1_totals['opex'], h1_totals['ebit']]
    colors = ['#1D9E75', '#FFC7CE', '#1D9E75', '#FFC7CE', '#2D2D2D']

    plt.figure(figsize=(8, 4))

    df_chart = pd.DataFrame({'metric': metrics, 'value': values, 'color': colors})
    sns.barplot(data=df_chart, x='metric', y='value', hue='metric',
                palette=dict(zip(metrics, colors)), legend=False)

    plt.title('H1 Profitability Waterfall: Healthy Flow-Through', fontweight='bold', pad=15)
    plt.axhline(0, color='black', linewidth=1)
    for i, v in enumerate(values):
        plt.text(i, v + (1000 if v>0 else -3000), f"{int(v):,}", ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig('chart_waterfall.png', dpi=150)
    plt.close()
    charts['waterfall'] = 'chart_waterfall.png'

    return charts # Added return statement


# ==========================================
# TASK 5: WRITE EXCEL REPORT
# ==========================================
def write_excel_report(pl_df, kpis, charts, output_path):
    wb = Workbook()

    # --- Sheet 1: Executive Summary ---
    ws_exec = wb.active
    ws_exec.title = 'Executive Summary'

    ws_exec.merge_cells('A1:B1')
    ws_exec['A1'] = 'H1 2024 Monthly Close Report'
    ws_exec['A1'].font = Font(bold=True, size=14, color='FFFFFF')
    ws_exec['A1'].fill = PatternFill(fill_type='solid', fgColor='1D9E75')
    ws_exec['A1'].alignment = Alignment(horizontal='center', vertical='center')

    GREEN_FILL = PatternFill(fill_type='solid', fgColor='C6EFCE')
    RED_FILL = PatternFill(fill_type='solid', fgColor='FFC7CE')

    row_idx = 3
    for k, v in kpis.items():
        ws_exec.cell(row=row_idx, column=1, value=k).font = Font(bold=True)
        cell_val = ws_exec.cell(row=row_idx, column=2, value=v)

        # Formatting & Conditional Colors
        if isinstance(v, (int, float)):
            if '%' in k:
                cell_val.number_format = '0.0"%"'
                if v > 0: cell_val.fill = GREEN_FILL
                elif v < 0: cell_val.fill = RED_FILL
            elif 'Revenue Growth' in k:
                cell_val.number_format = '0.0"%"'
                if v > 0: cell_val.fill = GREEN_FILL
                elif v < 0: cell_val.fill = RED_FILL
            else:
                cell_val.number_format = '#,##0;(#,##0)'
                if v > 0: cell_val.fill = GREEN_FILL
                elif v < 0: cell_val.fill = RED_FILL
        row_idx += 1

    ws_exec.freeze_panes = 'A3'
    ws_exec.column_dimensions['A'].width = 20
    ws_exec.column_dimensions['B'].width = 15

    # --- Sheet 2: Monthly P&L ---
    ws_pl = wb.create_sheet('Monthly P&L')
    headers = ['Month', 'Revenue', 'COGS', 'OpEx', 'Gross Profit', 'GP Margin %', 'EBIT', 'EBIT Margin %']
    ws_pl.append(headers)

    for col_num in range(1, len(headers)+1):
        cell = ws_pl.cell(row=1, column=col_num)
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill(fill_type='solid', fgColor='2D2D2D')

    for idx, row in pl_df.iterrows():
        ws_pl.append(row.tolist())
        current_row = ws_pl.max_row

        # Formatting
        for col_idx in [2, 3, 4, 5, 7]: # Money cols
            ws_pl.cell(row=current_row, column=col_idx).number_format = '#,##0;(#,##0)'
        for col_idx in [6, 8]: # Pct cols
            ws_pl.cell(row=current_row, column=col_idx).number_format = '0.0"%"'

        # H1 Total Row styling
        if row['month'] == 'H1 Total':
            thin_top = Border(top=Side(style='thin'))
            for c in range(1, len(headers)+1):
                ws_pl.cell(row=current_row, column=c).font = Font(bold=True)
                ws_pl.cell(row=current_row, column=c).border = thin_top

    ws_pl.column_dimensions['A'].width = 15
    for col_letter in ['B','C','D','E','F','G','H']:
        ws_pl.column_dimensions[col_letter].width = 13
    ws_pl.freeze_panes = 'B2'

    # --- Sheet 3: README ---
    ws_readme = wb.create_sheet('README')
    ws_readme['A1'], ws_readme['B1'] = 'Report Title:', 'H1 2024 Finance Report'
    ws_readme['A2'], ws_readme['B2'] = 'Author:', 'Lintang Permata Jati'
    ws_readme['A3'], ws_readme['B3'] = 'Generated:', datetime.date.today().strftime('%d %B %Y')
    ws_readme['A4'], ws_readme['B4'] = 'Data source:', 'gl_data.csv (Pipeline Output)'

    for r in range(1, 5): ws_readme[f'A{r}'].font = Font(bold=True)
    ws_readme.column_dimensions['A'].width = 16
    ws_readme.column_dimensions['B'].width = 30

    wb.save(output_path)


# ==========================================
# MASTER ORCHESTRATOR
# ==========================================
def run_monthly_close(gl_csv_path: str, output_path: str) -> str:
    print("=== MONTHLY CLOSE REPORT — H1 2024 ===")

    print("Stage 1/5: Ingesting...")
    df_raw = pd.read_csv(gl_csv_path)

    print("Stage 2/5: Cleaning...")
    df_clean = clean_transactions(df_raw)

    # Task 1 Verification Block
    print(f"  > Raw rows:    {len(df_raw)}")
    print(f"  > Clean rows:  {len(df_clean)}")
    print(f"  > Dropped:     {len(df_raw) - len(df_clean)}")
    print(f"  > Categories:  {df_clean['category'].value_counts().to_dict()}")
    print(f"  > Nulls:       {df_clean.isnull().sum().sum()}")
    print(f"  > Date range:  {df_clean['date'].min().date()} to {df_clean['date'].max().date()}")

    print("Stage 3/5: Computing P&L and KPIs...")
    pl = build_pl_summary(df_clean)
    kpis = compute_kpis(pl)
    print("  > KPIs Generated:")
    for k, v in kpis.items(): print(f"    - {k}: {v}")

    print("Stage 4/5: Building charts...")
    charts = build_charts(pl)

    print("Stage 5/5: Writing report...")
    write_excel_report(pl, kpis, charts, output_path)

    print(f"\n✓ Report complete: {output_path}")
    print(f"  Rows processed: {len(df_clean)}")
    print(f"  KPIs computed:  {len(kpis)}")
    print(f"  Charts saved:   {len(charts)}")
    return output_path

# ==========================================
# EXECUTE THE PIPELINE
# ==========================================
run_monthly_close('gl_data.csv', 'H1_2024_Monthly_Close.xlsx')

=== MONTHLY CLOSE REPORT — H1 2024 ===
Stage 1/5: Ingesting...
Stage 2/5: Cleaning...
  > Raw rows:    25
  > Clean rows:  24
  > Dropped:     1
  > Categories:  {'Revenue': 12, 'COGS': 6, 'OpEx': 6}
  > Nulls:       0
  > Date range:  2024-01-05 to 2024-06-28
Stage 3/5: Computing P&L and KPIs...
  > KPIs Generated:
    - H1 Revenue: 73400.0
    - H1 EBIT: 39300.0
    - Gross Margin %: 81.1
    - EBIT Margin %: 53.5
    - Best Month: 2024-06
    - Worst Month: 2024-02
    - Revenue Growth %: 46.1
Stage 4/5: Building charts...
Stage 5/5: Writing report...

✓ Report complete: H1_2024_Monthly_Close.xlsx
  Rows processed: 24
  KPIs computed:  7
  Charts saved:   3


'H1_2024_Monthly_Close.xlsx'